# Embedding Art - Quickstart

This notebook demonstrates the core workflow:
1. Load encoder and generator
2. Create a target concept from text/image/audio
3. Optimize a latent toward that concept
4. Visualize results

In [ ]:
import torch
from pathlib import Path
import matplotlib.pyplot as plt

# Verify MPS availability
print(f"MPS available: {torch.backends.mps.is_available()}")
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {DEVICE}")

## Setup

First, install ImageBind if you haven't:
```bash
git clone https://github.com/facebookresearch/ImageBind
cd ImageBind && pip install -e .
```

In [ ]:
from embedding_art import EmbeddingArtEngine, Concept, OptimizationConfig
from embedding_art.encoders.imagebind import ImageBindEncoder
from embedding_art.generators.image import SDXLImageGenerator
from embedding_art.regularizers import CompositeRegularizer

In [ ]:
# Load models (this will download weights on first run)
print("Loading ImageBind...")
encoder = ImageBindEncoder(device=DEVICE)

print("Loading SDXL VAE...")
generator = SDXLImageGenerator(device=DEVICE)

print("Done!")

In [ ]:
# Create engine
engine = EmbeddingArtEngine(encoder, device=DEVICE)
engine.register_generator("image", generator)

## Basic Usage: Text to Image

In [ ]:
# Create a concept from text
target = Concept.from_text("goldfish", encoder)
print(f"Target: {target}")
print(f"Embedding shape: {target.embedding.shape}")

In [ ]:
# Quick test run (fewer steps)
config = OptimizationConfig(
    steps=500,
    learning_rate=0.1,
    seed=42,
)

result = engine.optimize(
    target=target,
    output_modality="image",
    config=config,
)

In [ ]:
# View result
print(f"Final similarity: {result.final_similarity:.4f}")
print(f"Time: {result.elapsed_seconds:.1f}s")

img = result.get_final_image(generator)
plt.figure(figsize=(8, 8))
plt.imshow(img)
plt.axis('off')
plt.title(f"'goldfish' - similarity: {result.final_similarity:.4f}")
plt.show()

In [ ]:
# Plot optimization curve
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(result.loss_history)
ax1.set_xlabel('Step')
ax1.set_ylabel('Loss')
ax1.set_title('Total Loss')

ax2.plot(result.similarity_history)
ax2.set_xlabel('Step')
ax2.set_ylabel('Cosine Similarity')
ax2.set_title('Target Similarity')

plt.tight_layout()
plt.show()

## Concept Algebra

Combine multiple concepts using arithmetic operations.

In [ ]:
# Addition: combine concepts
fire = Concept.from_text("fire", encoder)
water = Concept.from_text("water", encoder)

fire_water = fire + water
print(f"Combined: {fire_water}")

In [ ]:
# Weighted combination
ocean = Concept.from_text("ocean", encoder)
sunset = Concept.from_text("sunset", encoder)

combined = 0.7 * ocean + 0.3 * sunset
print(f"Weighted: {combined}")

In [ ]:
# Subtraction: remove attributes
dog = Concept.from_text("dog", encoder)
fur = Concept.from_text("fur", encoder)

hairless_dog = dog - 0.3 * fur
print(f"Subtracted: {hairless_dog}")

In [ ]:
# Spherical interpolation
goldfish = Concept.from_text("goldfish", encoder)
flamingo = Concept.from_text("flamingo", encoder)

midpoint = Concept.slerp(goldfish, flamingo, t=0.5)
print(f"Midpoint: {midpoint}")

In [ ]:
# Optimize toward combined concept
result_combined = engine.optimize(
    target=0.6 * ocean + 0.4 * fire,
    output_modality="image",
    config=OptimizationConfig(steps=500, seed=42),
)

img = result_combined.get_final_image(generator)
plt.figure(figsize=(8, 8))
plt.imshow(img)
plt.axis('off')
plt.title(f"'0.6*ocean + 0.4*fire' - similarity: {result_combined.final_similarity:.4f}")
plt.show()

## Regularization Effects

Compare different regularization strengths.

In [ ]:
target = Concept.from_text("galaxy", encoder)

# Minimal regularization (more raw/adversarial)
result_minimal = engine.optimize(
    target=target,
    output_modality="image",
    config=OptimizationConfig(steps=500, seed=42),
    regularizers=CompositeRegularizer.minimal(),
)

# Heavy regularization (more coherent)
result_heavy = engine.optimize(
    target=target,
    output_modality="image",
    config=OptimizationConfig(steps=500, seed=42),
    regularizers=CompositeRegularizer.heavy(),
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7))

ax1.imshow(result_minimal.get_final_image(generator))
ax1.set_title(f"Minimal reg - sim: {result_minimal.final_similarity:.4f}")
ax1.axis('off')

ax2.imshow(result_heavy.get_final_image(generator))
ax2.set_title(f"Heavy reg - sim: {result_heavy.final_similarity:.4f}")
ax2.axis('off')

plt.tight_layout()
plt.show()

## Interpolation Series

Generate outputs along a path between two concepts.

In [ ]:
# This takes a while - generates 5 images
concept_a = Concept.from_text("goldfish", encoder)
concept_b = Concept.from_text("flamingo", encoder)

results = engine.interpolation_series(
    concept_a=concept_a,
    concept_b=concept_b,
    output_modality="image",
    steps=5,
    config=OptimizationConfig(steps=300),
)

In [ ]:
# Display interpolation
fig, axes = plt.subplots(1, len(results), figsize=(4*len(results), 4))

for i, (ax, result) in enumerate(zip(axes, results)):
    t = i / (len(results) - 1)
    img = result.get_final_image(generator)
    ax.imshow(img)
    ax.set_title(f"t={t:.2f}")
    ax.axis('off')

plt.tight_layout()
plt.show()

## Cross-Modal: Audio to Image

If you have an audio file, you can optimize an image toward it.

In [ ]:
# Example with audio (uncomment if you have an audio file)
# audio_concept = Concept.from_audio("path/to/thunder.wav", encoder)
# 
# result_audio = engine.optimize(
#     target=audio_concept,
#     output_modality="image",
#     config=OptimizationConfig(steps=500),
# )
#
# img = result_audio.get_final_image(generator)
# plt.imshow(img)
# plt.title("What does thunder look like?")
# plt.axis('off')
# plt.show()

## Save Results

In [ ]:
# Save image
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

img = result.get_final_image(generator)
img.save(output_dir / "goldfish_optimized.png")
print(f"Saved to {output_dir / 'goldfish_optimized.png'}")

# Save embedding for later use
target.save(output_dir / "goldfish_embedding.pt")
print(f"Saved embedding to {output_dir / 'goldfish_embedding.pt'}")

In [ ]:
# Load saved embedding
loaded = Concept.load(output_dir / "goldfish_embedding.pt")
print(f"Loaded: {loaded}")
print(f"Similarity to original: {target.similarity(loaded):.4f}")